# 📦 1. 미니 테스트 파이프라인 코드 (셀 분리형)
본격적인 데이터 작업 전에 전체적인 하드웨어 및 라이브러리 호환성을 5~10분 내로 가볍게 검증하기 위한 파일럿 코드입니다. 원본 데이터셋에서 일부만 추출하여 학습 및 저장 전 과정을 모의 테스트합니다.

# [셀 1] 환경 세팅 및 드라이브 마운트

In [1]:
# ==========================================
# [셀 1] 라이브러리 설치 및 작업 경로 설정
# ==========================================
# 💡 --no-deps "trl<0.9.0" 부분을 제거하고 최신 버전으로 설치합니다.
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft accelerate bitsandbytes datasets

from google.colab import drive
import os

drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/2차인콘/모델학습"
os.makedirs(BASE_PATH, exist_ok=True)
print(f"✅ 작업 폴더 세팅 완료: {BASE_PATH}")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ps7sdc8b/unsloth_0c244d4dc5ba44048904d44993f79b49
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ps7sdc8b/unsloth_0c244d4dc5ba44048904d44993f79b49
  Resolved https://github.com/unslothai/unsloth.git to commit e25e7895a5024b3545d22b334c00b468b0f28141
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 152.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 22.9 MB/s eta 0:00:00
  

# [셀 2] 미니 테스트 실행 (Train 10,000건 / Eval 1,000건)

In [ ]:
# ==========================================
# [셀 2] 1만 건 규모 미니 샘플링 학습 테스트 (완벽 수정본)
# ==========================================
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig # 💡 핵심 수정 1: SFTConfig 불러오기

BASE_PATH = "/content/drive/MyDrive/2차인콘/모델학습"
max_seq_length = 2048
dtype = None
load_in_4bit = True

print("1️⃣ 모델 및 토크나이저 초기화...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("2️⃣ LoRA 레이어 설정...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("3️⃣ 원본 파일 기반 임시 샘플링...")
train_dataset = load_dataset("json", data_files=f"{BASE_PATH}/qwen_train_data.jsonl", split="train")
eval_dataset = load_dataset("json", data_files=f"{BASE_PATH}/qwen_test_data.jsonl", split="train")

if len(train_dataset) > 10000: train_dataset = train_dataset.shuffle(seed=42).select(range(10000))
if len(eval_dataset) > 1000: eval_dataset = eval_dataset.shuffle(seed=42).select(range(1000))

def format_chatml(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)}

train_dataset = train_dataset.map(format_chatml)
eval_dataset = eval_dataset.map(format_chatml)

print("4️⃣ 파라미터 빌드 및 모의 학습 시작...")
# 💡 핵심 수정 2: TrainingArguments 대신 SFTConfig 사용 (파라미터 위치 최적화)
sft_config = SFTConfig(
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 30,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    eval_strategy = "steps",
    eval_steps = 15,
    optim = "adamw_8bit",
    seed = 3407,
    output_dir = f"{BASE_PATH}/outputs_mini_test",
)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = sft_config, # 교체된 설정 객체 투입
)

trainer.train()

print("5️⃣ 가중치 저장 테스트...")
model.save_pretrained(f"{BASE_PATH}/Banky_Model_mini_test")
tokenizer.save_pretrained(f"{BASE_PATH}/Banky_Model_mini_test")
print("🎉 [성공] 미니 파이프라인이 완벽하게 작동합니다. 본 학습 준비 가능!")

1️⃣ 모델 및 토크나이저 초기화...
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: pad_token was a vision token (<|vision_pad|>) on a text-only model. Replaced with <|endoftext|> to avoid NaN losses.


2️⃣ LoRA 레이어 설정...
3️⃣ 원본 파일 기반 임시 샘플링...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

...

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
15,1.360575,1.237157
30,1.074096,1.118932


5️⃣ 가중치 저장 테스트...


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/2차인콘/모델학습/Banky_Model_mini_test/tokenizer_config.json.


🎉 [성공] 미니 파이프라인이 완벽하게 작동합니다. 본 학습 준비 가능!


# ✂️ 2. 고급 데이터 다이어트 및 재배치 스크립트
제안하신 아이디어를 코드로 완벽하게 구현했습니다. 원본 테스트 데이터셋(qwen_test_data.jsonl)을 무작위로 뒤섞은 후, 실시간 검증용(2,000건)과 최종 수능 시험지(4,000건)를 안전하게 격리합니다. 이후 남은 테스트 데이터 중에서 말뭉치(번역)를 제외한 순수 '금융상담 데이터'만 필터링하여 훈련 데이터셋으로 흡수시키는 고도화 전처리 코드입니다.

In [ ]:
import json
import random

BASE_PATH = "/content/drive/MyDrive/2차인콘/모델학습"
train_input = f"{BASE_PATH}/qwen_train_data.jsonl"
test_input = f"{BASE_PATH}/qwen_test_data.jsonl"

# 출력될 고품질 최적화 파일 경로
train_output = f"{BASE_PATH}/qwen_train_data_diet.jsonl"
val_output = f"{BASE_PATH}/qwen_val_data_fixed.jsonl"
final_test_output = f"{BASE_PATH}/qwen_final_test_data.jsonl"

print("🔍 1. 원본 훈련 데이터 분석 및 분리 프로세스...")
cs_train_pool = []
corpus_train_pool = []

with open(train_input, 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line.strip())
        user_text = next((msg["content"] for msg in item["messages"] if msg["role"] == "user"), "")
        if "[대상언어]" in user_text:
            corpus_train_pool.append(item)
        else:
            cs_train_pool.append(item)

print(f" -> 오리지널 상담 데이터: {len(cs_train_pool)}건 (100% 보존)")
print(f" -> 오리지널 번역 말뭉치: {len(corpus_train_pool)}건")

# 말뭉치 데이터 핵심 액기스 12만 건 무작위 샘플링
TARGET_CORPUS_SIZE = 120000
if len(corpus_train_pool) > TARGET_CORPUS_SIZE:
    sampled_corpus = random.sample(corpus_train_pool, TARGET_CORPUS_SIZE)
else:
    sampled_corpus = corpus_train_pool
print(f" -> 말뭉치 스마트 다이어트 완료: {len(corpus_train_pool)}건 -> {len(sampled_corpus)}건")


print("\n🔮 2. 테스트 데이터 분할 및 자원 회수 프로세스...")
with open(test_input, 'r', encoding='utf-8') as f:
    test_rows = [json.loads(line.strip()) for line in f]

# 데이터 편향 방지를 위한 전방위 셔플
random.seed(3407)
random.shuffle(test_rows)

# 고정 시험지 분할 (중간 검증용 2000건, 최종 평가용 4000건 격리)
VAL_SIZE = 2000
FINAL_TEST_SIZE = 4000

val_set = test_rows[:VAL_SIZE]
final_test_set = test_rows[VAL_SIZE : VAL_SIZE + FINAL_TEST_SIZE]
remaining_test_set = test_rows[VAL_SIZE + FINAL_TEST_SIZE:]

print(f" -> [격리 완료] 실시간 검증용 고정 데이터(Val): {len(val_set)}건")
print(f" -> [격리 완료] 오프라인 제출용 최종 시험지(Final Test): {len(final_test_set)}건")

# 남은 테스트 데이터 중 '금융상담 데이터'만 필터링하여 회수
recovered_cs_count = 0
for item in remaining_test_set:
    user_text = next((msg["content"] for msg in item["messages"] if msg["role"] == "user"), "")
    if "[대상언어]" not in user_text: # 말뭉치가 아닌 순수 금융상담 케이스
        cs_train_pool.append(item)
        recovered_cs_count += 1

print(f" -> [자원 회수] 미사용 테스트 풀에서 순수 금융상담 데이터 {recovered_cs_count}건을 찾아내어 Train으로 합류시켰습니다!")


print("\n💾 3. 최종 데이터셋 정제 및 영구 저장...")
# 모든 요소 융합 후 최종 셔플
final_train_set = cs_train_pool + sampled_corpus
random.shuffle(final_train_set)

# 파일 쓰기
with open(train_output, 'w', encoding='utf-8') as f:
    for item in final_train_set: f.write(json.dumps(item, ensure_ascii=False) + '\n')
with open(val_output, 'w', encoding='utf-8') as f:
    for item in val_set: f.write(json.dumps(item, ensure_ascii=False) + '\n')
with open(final_test_output, 'w', encoding='utf-8') as f:
    for item in final_test_set: f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"📈 [최종 스탯] 학습 데이터셋 정제 완료: 총 {len(final_train_set)}건")
print("✅ 모든 정제 파일이 드라이브 경로에 안전하게 빌드되었습니다.")

🔍 1. 원본 훈련 데이터 분석 및 분리 프로세스...
 -> 오리지널 상담 데이터: 80000건 (100% 보존)
 -> 오리지널 번역 말뭉치: 491637건
 -> 말뭉치 스마트 다이어트 완료: 491637건 -> 120000건

🔮 2. 테스트 데이터 분할 및 자원 회수 프로세스...
 -> [격리 완료] 실시간 검증용 고정 데이터(Val): 2000건
 -> [격리 완료] 오프라인 제출용 최종 시험지(Final Test): 4000건
 -> [자원 회수] 미사용 테스트 풀에서 순수 금융상담 데이터 9163건을 찾아내어 Train으로 합류시켰습니다!

💾 3. 최종 데이터셋 정제 및 영구 저장...
📈 [최종 스탯] 학습 데이터셋 정제 완료: 총 209163건
✅ 모든 정제 파일이 드라이브 경로에 안전하게 빌드되었습니다.


# 🚀 3. 본 학습 풀악셀 파인튜닝 코드
위 전처리 과정을 통해 뼈대와 내실을 모두 꽉 채운 qwen_train_data_diet.jsonl(약 20만 건 이상) 및 고정 검증셋 qwen_val_data_fixed.jsonl을 사용하여 실제 모델을 굽는 최종 풀 파이프라인입니다.

실시간으로 200스텝마다 2,000건 고정 시험지로 점수를 체크하며, 코랩 불안정성에 대처하기 위해 500스텝마다 구글 드라이브에 자동으로 안전 저장 장치(체크포인트)를 활성화합니다.

In [2]:
# ==========================================
# [셀 3] 뱅키(Banky) 최종 본 학습 풀악셀 파이프라인 (완벽 수정본)
# ==========================================
import torch
import os
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig # 💡 SFTConfig 적용
from transformers.trainer_utils import get_last_checkpoint # 💡 자동 복구 기능

BASE_PATH = "/content/drive/MyDrive/2차인콘/모델학습"
max_seq_length = 2048
dtype = None
load_in_4bit = True

print("1️⃣ 대형 금융 언어 모델 로드 (4-bit 최적화 환경)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("2️⃣ 다국어 및 상담 융합을 위한 LoRA 구조 어댑션...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("3️⃣ 다이어트 및 회수 작업이 완료된 청정 데이터셋 로드...")
train_dataset = load_dataset("json", data_files=f"{BASE_PATH}/qwen_train_data_diet.jsonl", split="train")
eval_dataset = load_dataset("json", data_files=f"{BASE_PATH}/qwen_val_data_fixed.jsonl", split="train")

def format_chatml(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)}

print(" -> 대화형 ChatML 포맷 매핑 진행...")
train_dataset = train_dataset.map(format_chatml)
eval_dataset = eval_dataset.map(format_chatml)

print(f"📊 [최종 확인] 학습 진행 건수: {len(train_dataset)}건 / 실시간 검증 건수: {len(eval_dataset)}건")

print("4️⃣ 초고속 하이퍼파라미터 구성 및 트레이너 초기화...")
# 💡 핵심 수정: SFTConfig로 파라미터 통합
sft_config = SFTConfig(
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 50,
    num_train_epochs = 1,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    eval_strategy = "steps",
    eval_steps = 1500,
    save_strategy = "steps",
    save_steps = 1500,
    save_total_limit = 2,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = f"{BASE_PATH}/outputs_full_run",
)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer, # 💡 문법 업데이트 반영
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = sft_config,
)

print("🚀 글로벌 금융 AI 대화 솔루션 본 학습 풀악셀 스타트!")

# 💡 자동 복구(Resume) 로직 탑재 완료
latest_checkpoint = get_last_checkpoint(sft_config.output_dir)

if latest_checkpoint is not None:
    print(f"🔄 끊긴 지점 발견! 이전에 저장된 체크포인트({latest_checkpoint})부터 안전하게 이어서 학습을 재개합니다!")
    trainer_stats = trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("✨ 저장된 내역이 없습니다. 처음부터 새롭게 학습을 시작합니다!")
    trainer_stats = trainer.train()

print("5️⃣ 영구 보존용 최종 가중치 어댑터 동기화...")
final_save_path = f"{BASE_PATH}/Banky_Model_Final"
model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)

print(f"🎉 [완료] AI 챗봇 '뱅키(Banky)' 최종 두뇌 엔진이 드라이브에 안착했습니다!")
print(f"💾 최종 위치: {final_save_path}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
1️⃣ 대형 금융 언어 모델 로드 (4-bit 최적화 환경)...
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

...

Step,Training Loss,Validation Loss
7200,0.837265,0.779703
7400,0.814956,0.778450
7600,0.812773,0.775890
7800,0.834509,0.776106
8000,0.757044,0.772658
8200,0.808498,0.773407
8400,0.767656,0.771042
8600,0.806079,0.769790
8800,0.769991,0.768188
9000,0.823926,0.767855


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/2차인콘/모델학습/outputs_full_run/checkpoint-20000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/2차인콘/모델학습/outputs_full_run/checkpoint-20500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/2차인콘/모델학습/outputs_full_run/checkpoint-21000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/2차인콘/모델학습/outputs_full_run/checkpoint-21500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/2차인콘/모델학습/outputs_full_run/checkpoint-22000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/2차인콘/모델학습/outputs_full_run/checkpoint-22500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/2차인콘/모델학습/outputs_full_run/checkpoint-23000/tokenizer_config.json.
Unsloth: Rest

5️⃣ 영구 보존용 최종 가중치 어댑터 동기화...


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/2차인콘/모델학습/Banky_Model_Final/tokenizer_config.json.


🎉 [완료] AI 챗봇 '뱅키(Banky)' 최종 두뇌 엔진이 드라이브에 안착했습니다!
💾 최종 위치: /content/drive/MyDrive/2차인콘/모델학습/Banky_Model_Final


# 4. 최종 테스트 코드

In [3]:
# ==========================================
# [셀 4] 뱅키(Banky) 최종 수능 테스트 (추론 및 성능 검증)
# ==========================================
from unsloth import FastLanguageModel
import json
import random

BASE_PATH = "/content/drive/MyDrive/2차인콘/모델학습"
final_model_path = f"{BASE_PATH}/Banky_Model_Final"
test_file_path = f"{BASE_PATH}/qwen_final_test_data.jsonl"

print("1️⃣ 완성된 뱅키 모델 불러오는 중...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = final_model_path,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model) # 💡 Unsloth의 2배 빠른 초고속 추론 모드 켜기!

print("2️⃣ 최종 평가용(수능) 데이터 4,000건 로드 중...")
with open(test_file_path, 'r', encoding='utf-8') as f:
    test_data = [json.loads(line) for line in f]

# 무작위로 1개 뽑아서 테스트해보기 (이 셀을 반복 실행하면 매번 다른 질문을 테스트합니다)
sample = random.choice(test_data)
messages = sample["messages"]

# 정답(assistant) 부분을 제외하고 모델에게 던져줄 질문만 추출
prompt_messages = [msg for msg in messages if msg["role"] != "assistant"]
ground_truth = next((msg["content"] for msg in messages if msg["role"] == "assistant"), "")

# ChatML 템플릿 적용 및 텐서 변환
prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

print("\n🤖 [뱅키가 답변을 생성하고 있습니다...]\n")
outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)
response = tokenizer.batch_decode(outputs)[0]

# 프롬프트 부분 잘라내고 순수 답변만 추출
answer = response.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

print("=" * 60)
print(f"👤 [고객 질문]\n{prompt_messages[-1]['content']}\n")
print("-" * 60)
print(f"🎯 [실제 정답 (Ground Truth)]\n{ground_truth}\n")
print("-" * 60)
print(f"🏦 [뱅키의 답변 (AI)]\n{answer}\n")
print("=" * 60)

1️⃣ 완성된 뱅키 모델 불러오는 중...
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: pad_token was a vision token (<|vision_pad|>) on a text-only model. Replaced with <|endoftext|> to avoid NaN losses.


2️⃣ 최종 평가용(수능) 데이터 4,000건 로드 중...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🤖 [뱅키가 답변을 생성하고 있습니다...]



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


👤 [고객 질문]
최근 교통카드 환급 내역을 확인할 수 있나요?
다음 달 환급 일정도 미리 알 수 있나요?

[요청]: 교통카드 환급 내역과 다음 달 환급 일정 확인 여부를 안내하시오.

------------------------------------------------------------
🎯 [실제 정답 (Ground Truth)]
최근 교통카드 환급 내역은 각각 입금된 것으로 확인되며, 환급은 카드사 정산 일정에 따라 자동으로 반영됩니다. 다음 달 환급일은 고정되어 있지 않고 카드사 정산 주기에 따라 달라질 수 있으니, 필요하시면 같은 기간의 다른 교통카드 거래도 추가로 조회해 드리겠습니다.

------------------------------------------------------------
🏦 [뱅키의 답변 (AI)]
교통카드 환급 내역은 최근 3개월 이내에 발생한 금액만 조회 가능하며, 기간을 알려주시면 정확히 확인해 드립니다. 현재 시스템에서는 다음 달 환급 일정을 확인하지 못하므로, 다음 달 환급이 예정된 경우에만 조회가 가능합니다.



In [5]:
# ==========================================
# [셀 5] 뱅키(Banky) 다국어 번역 능력 테스트
# ==========================================
test_sentences = [
    ("고객님의 계좌 비밀번호가 3회 오류 초과되어 잠금 처리되었습니다. 영업점을 방문해 주시기 바랍니다.", "영어"),
    ("해외 송금 수수료는 송금액의 1%이며, 전신료 8,000원이 별도로 부과됩니다.", "중국어(간체)"),
    ("OTP 배터리가 방전된 경우 모바일 뱅킹에서 디지털 OTP로 즉시 전환 발급이 가능합니다.", "베트남어")
]

print("🌍 [뱅키 다국어 번역 모듈 가동 중...]\n")

for text, lang in test_sentences:
    # 전처리 코드에 맞춰 프롬프트 세팅
    prompt_content = f"다음 한국어 금융 문장을 정확하게 번역하세요:\n{text}\n[대상언어]: {lang}"
    messages = [
        {"role": "system", "content": "당신은 인사이트뱅크의 친절하고 전문적인 AI 은행원 '뱅키'입니다. 고객의 질문에 정확하고 알기 쉽게 답변해 주세요."},
        {"role": "user", "content": prompt_content}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
    response = tokenizer.batch_decode(outputs)[0]
    answer = response.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

    print("=" * 60)
    print(f"🇰🇷 [원문]: {text}")
    print(f"🌐 [목표 언어]: {lang}")
    print("-" * 60)
    print(f"🤖 [뱅키 번역]: {answer}")
print("=" * 60)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🌍 [뱅키 다국어 번역 모듈 가동 중...]



Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🇰🇷 [원문]: 고객님의 계좌 비밀번호가 3회 오류 초과되어 잠금 처리되었습니다. 영업점을 방문해 주시기 바랍니다.
🌐 [목표 언어]: 영어
------------------------------------------------------------
🤖 [뱅키 번역]: Due to three consecutive errors, your account password has been locked. Please visit the branch office.


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🇰🇷 [원문]: 해외 송금 수수료는 송금액의 1%이며, 전신료 8,000원이 별도로 부과됩니다.
🌐 [목표 언어]: 중국어(간체)
------------------------------------------------------------
🤖 [뱅키 번역]: 海外汇款手续费为汇款金额的1%，另外收取8000韩元的电报费。
🇰🇷 [원문]: OTP 배터리가 방전된 경우 모바일 뱅킹에서 디지털 OTP로 즉시 전환 발급이 가능합니다.
🌐 [목표 언어]: 베트남어
------------------------------------------------------------
🤖 [뱅키 번역]: Nếu pin OTP bị cạn pin, bạn có thể chuyển đổi và cấp phát OTP kỹ thuật số ngay lập tức trên ứng dụng ngân hàng di động.
